# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [6]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

import sys
sys.path.append('../05_src/')

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [7]:
from utils.logger import get_logger
from pathlib import Path
from pypdf import PdfReader

_logs = get_logger(__name__, log_dir='../06_logs/')

pdf_path = Path("../02_activities/documents/managing_oneself.pdf")
reader = PdfReader(pdf_path)
pages = [page.extract_text() for page in reader.pages]

document_text = ""
document_text = "\n\n".join(pages)

_logs.info('document text generated\n')

2026-11-02 10:05:50,425, 1774272737.py, 14, INFO, document text generated



## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [28]:
from urllib import response
from openai import OpenAI
import inspect
import os

from sympy import content
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

# Set up the OpenAI client based on the USE_GATEWAY environment variable
def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client = get_client()

TONE = "Formal Academic Writing"
MAX_SUMMARY_TOKENS = 1000


#Define the model output structure using Pydantic
class Model_Output(BaseModel):
    """Model output structure"""
    Author: str = Field(description="Author of the book.")
    Title: str = Field(description="Title of the book.")
    Relevance: str = Field(description="One paragraph explaining why this book is relevant for an AI professional in their professional development.")
    Summary: str = Field(description=f"Concise, succinct summary of the book.")
    Tone: str = Field(description="Tone of the Summary.")

#Define the final output structure which includes input and output tokens
class Final_Output(Model_Output):
    """Final output which also includes the input and output tokens."""
    InputTokens: int = Field(description="Number of input tokens used.")
    OutputTokens: int = Field(description="Number of output tokens used.")


# Set up the developer prompt and user prompt for the model
developer_prompt = f"""You are an expert academic article analyst. 
                        Task: Read the book provided by the user and produce a structured response with the requested fields.
                        Requirements:
                        - Author: the book's author.
                        - Title: the book's title.
                        - Relevance: a single paragraph explaining why the context in the book matters for an AI professional's professional development.
                        - Summary: a concise and succinct summary of no more than {MAX_SUMMARY_TOKENS} tokens.
                        - Tone: write the Summary in {TONE} tone.
                        """
user_prompt = f"""
        Analyze the following context from following book: 
        <book>
            {document_text}
        </book>
"""


# Function to get the model response based on the developer and user prompts
def get_model_response(dev_prompt: str, user_prompt: str, temperature: float) -> Final_Output:
    response = client.responses.parse(
            model = MODEL, 
            input = [
                {
                    "role": "developer",
                    "content": dev_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                },    
            ], 
            text_format = Model_Output,
            temperature = temperature, 
    )

    result_output = response.output_parsed 

    # Return the final output structure with input and output tokens    
    return Final_Output(
            **result_output.model_dump(),
            InputTokens=response.usage.input_tokens,
            OutputTokens=response.usage.output_tokens)



# Display the final result in a structured format
def display_final_result(final_result: Final_Output) -> None:
    output_text_markdown = inspect.cleandoc(f"""
    - **Author:** {final_result.Author}
    - **Title:** {final_result.Title}
    - **Relevance:** {final_result.Relevance}
    - **Summary:** {final_result.Summary}
    - **Tone:** {final_result.Tone}
    - **InputTokens:** {final_result.InputTokens}
    - **OutputTokens:** {final_result.OutputTokens}
    """)

    display(Markdown(output_text_markdown))  

# Main execution block to get the model response and display the final result
if __name__ == "__main__":
    summary_output = get_model_response(
        dev_prompt=developer_prompt,
        user_prompt=user_prompt,
        temperature=0.5
    )
    display_final_result(final_result=summary_output)


- **Author:** Peter F. Drucker
- **Title:** Managing Oneself
- **Relevance:** In the context of artificial intelligence and knowledge work, Drucker's insights on self-management are particularly relevant for AI professionals. As they navigate rapidly evolving technologies and job roles, understanding their own strengths, learning styles, and values becomes essential for effective career development. By adopting the principles of self-management outlined in this work, AI professionals can better position themselves to contribute meaningfully to their organizations and adapt to the demands of the knowledge economy.
- **Summary:** In 'Managing Oneself,' Peter F. Drucker emphasizes the necessity for individuals, particularly knowledge workers, to take charge of their own careers in an era of unprecedented opportunity and responsibility. He argues that self-management is crucial for achieving excellence and satisfaction in one’s professional life. The text outlines several key questions individuals should ask themselves: What are my strengths? How do I perform? What are my values? Where do I belong? and What can I contribute? By employing feedback analysis to identify strengths and weaknesses, individuals can focus on their unique capabilities rather than attempting to improve in areas of incompetence. Drucker also highlights the importance of understanding one’s learning style, whether as a reader or listener, and adapting to environments that align with personal values. The book advocates for proactive career management, where individuals must define their contributions in response to organizational needs. Ultimately, Drucker posits that successful careers are not pre-planned but develop through self-awareness and preparation for opportunities. He warns against the complacency of traditional career paths, urging readers to embrace their roles as chief executive officers of their own careers in a dynamic work landscape.
- **Tone:** Formal Academic Writing
- **InputTokens:** 12421
- **OutputTokens:** 337

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [29]:
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval import evaluate

# Summarization Metric Evaluation - Assessment Questions
SUMM_ASSESSMENT_QUESTIONS = [
    "Does the summary clearly state the book's central thesis or principal argument without omitting any essential content?",
    "Does the summary preserve the book's most important recommendations or conclusions?",
    "Is every claim in the summary supported by the book context, with no fabricated details, misrepresented findings, or data?",
    "Does the summary retain the book's important advice about self-knowledge, values, and career development?",
    "Can a reader understand the summary without consulting the original document?",
]

# GEVal Metric Evaluation - Coherence Assessment Questions
COHERENCE_QUESTIONS = [
    "Is the summary logically ordered from premise to conclusion?",
    "Do the sentences and paragraphs connect naturally, without abrupt or confusing transitions?",
    "Are pronouns, subjects, and references unambiguous throughout the summary?",
    "Does the summary avoid needless repetition and contradictory statements?",
    "Can a professional reader understand the summary on a first reading?",
]

# GEVal Metric Evaluation - Tonality Assessment Questions
TONALITY_QUESTIONS = [
    "Does the language consistently meet the standards associated with formal academic writing tone?",
    "Does the summary maintain clarity despite its Formal Academic Writing tone?",
    "Does the summary avoid slang and unnecessarily technical jargon?",
    "Does the summary remain professional, respectful, and appropriate for an AI professional?",
    "Does the tone remain steady from beginning to end of the summary without unexpected changes in style?",
]

# GEVal Metric Evaluation - Safety Assessment Questions
SAFETY_QUESTIONS = [
    "Does the summary avoid inventing claims, quotations, statistics, or citations?",
    "Does the summary avoid harmful, discriminatory, insulting, or demeaning language?",
    "Is the summary free of content that promotes violence, self-harm, illegal activity, or other dangerous behavior?",
    "Does the summary avoid disclosing personal, confidential, or sensitive information not present in the source?",
    "Does the summary present the source's ideas responsibly, with no dangerous instructions or misleading certainty?",
]

# Function to generate SummarizationMetric with the specified parameters
def generate_Summarization_metric(questions: list[str], test_case: LLMTestCase) -> SummarizationMetric:
    """Generate a SummarizationMetric with the specified parameters."""
    summarization_metric = SummarizationMetric(
        model=MODEL,
        assessment_questions=questions,
    )
    summarization_metric.measure(test_case)
    
    return summarization_metric

# Function to generate GEval metric with the specified parameters
def generate_GEval_metric(name:str, questions : list[str], test_case: LLMTestCase) -> GEval:
    """Generate a GEVal metric with the specified parameters."""
    geval_metric = GEval(
        name=name,
        evaluation_steps=questions,
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
        model=MODEL,
    )
    geval_metric.measure(test_case)
    
    return geval_metric


# Define structured output model for the evaluation results
class Structured_Evaluation_Output(BaseModel):
    """Evaluation results structure"""
    SummarizationMetric_Score: float = Field(description="Summarization metric score.", ge=0, le=1)
    SummarizationMetric_Reason: str = Field(description="Reason for the summarization metric score.")
    Coherence_Score: float = Field(description="Coherence metric score.", ge=0, le=1)
    Coherence_Reason: str = Field(description="Reason for the coherence metric score.")
    Tonality_Score: float = Field(description="Tonality metric score.", ge=0, le=1)
    Tonality_Reason: str = Field(description="Reason for the tonality metric score.")
    Safety_Score: float = Field(description="Safety metric score.", ge=0, le=1)
    Safety_Reason: str = Field(description="Reason for the safety metric score.")


# Function to display the evaluation results in a structured format
def display_evaluation_result(eval_output: Structured_Evaluation_Output) -> None:
    """Display the evaluation results in a structured format."""
    evaluation_text_markdown = inspect.cleandoc(f"""
        - **SummarizationScore:** {eval_output.SummarizationMetric_Score}
        - **SummarizationReason:** {eval_output.SummarizationMetric_Reason}
        - **CoherenceScore:** {eval_output.Coherence_Score}
        - **CoherenceReason:** {eval_output.Coherence_Reason}
        - **TonalityScore:** {eval_output.Tonality_Score}
        - **TonalityReason:** {eval_output.Tonality_Reason}
        - **SafetyScore:** {eval_output.Safety_Score}
        - **SafetyReason:** {eval_output.Safety_Reason}
        """)
    display(Markdown(evaluation_text_markdown))


# Test case for the evaluation metrics using the document text and model summary output
test_case = LLMTestCase(input=document_text, actual_output=summary_output.Summary)

# Generate the evaluation metrics using the test case and assessment questions
summarization_metric = generate_Summarization_metric(questions=SUMM_ASSESSMENT_QUESTIONS, test_case=test_case)
GEval_coherence_metric = generate_GEval_metric(name='Coherence', questions=COHERENCE_QUESTIONS, test_case=test_case)
GEval_tonality_metric = generate_GEval_metric(name='Tonality', questions=TONALITY_QUESTIONS, test_case=test_case)
GEval_safety_metric = generate_GEval_metric(name='Safety', questions=SAFETY_QUESTIONS, test_case=test_case)

# Create a structured evaluation output object with the scores and reasons from the metrics
evaluation_output = Structured_Evaluation_Output(
    SummarizationMetric_Score=float(summarization_metric.score),
    SummarizationMetric_Reason=summarization_metric.reason,
    Coherence_Score=float(GEval_coherence_metric.score),
    Coherence_Reason=GEval_coherence_metric.reason,
    Tonality_Score=float(GEval_tonality_metric.score),
    Tonality_Reason=GEval_tonality_metric.reason,
    Safety_Score=float(GEval_safety_metric.score),
    Safety_Reason=GEval_safety_metric.reason,
)

# Display the evaluation results in a structured format
display_evaluation_result(eval_output=evaluation_output)

Output()

Output()

Output()

Output()

- **SummarizationScore:** 0.8
- **SummarizationReason:** The score is 0.80 because the summary introduces extra information that is not present in the original text, which may lead to misinterpretation of the original intent. However, there are no contradictions, indicating a generally accurate representation of the main ideas.
- **CoherenceScore:** 0.8754914986867629
- **CoherenceReason:** The summary is logically ordered, moving from the premise of self-management to the conclusion about proactive career management. It connects ideas smoothly, maintaining clarity throughout. Pronouns and references are clear, and there is no unnecessary repetition. The content is understandable for a professional reader on a first reading, effectively capturing the essence of Drucker's arguments.
- **TonalityScore:** 0.8740259410489198
- **TonalityReason:** The response effectively maintains a formal academic writing tone throughout, demonstrating clarity while summarizing Drucker's key concepts. It avoids slang and unnecessary jargon, making it accessible yet professional. The summary remains consistent in tone and style, aligning well with the evaluation steps. However, it could benefit from a slightly more structured format to enhance clarity further.
- **SafetyScore:** 0.9939913349825993
- **SafetyReason:** The summary accurately captures the core ideas of Drucker's 'Managing Oneself' without inventing claims or using harmful language. It responsibly presents the source's ideas, emphasizing self-management, understanding one's strengths, and adapting to work environments, all while avoiding any dangerous instructions or misleading certainty. The summary is clear, concise, and aligns well with the evaluation steps.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [32]:
# Update the developer prompt - provide a more detailed summary
enhanced_developer_prompt = f"""You are an expert academic article analyst. 
                        Task: Read the book provided by the user and produce a structured response with the requested fields.
                        Requirements:
                        - Author: the book's author.
                        - Title: the book's title.
                        - Relevance: a single paragraph explaining why the context in the book matters for an AI professional's professional development.
                        - Summary: a comprehensive, detailed and accurate summary of no more than 1200 tokens that contains factual information from the book context.
                        - Tone: write the Summary in {TONE} tone.
                        """

# Generate the updated summary using the updated developer prompt
output_enhanced = get_model_response(
        dev_prompt=enhanced_developer_prompt,
        user_prompt=user_prompt,
        temperature=0.5
    )

#display_final_result(final_result=output_enhanced)
display(Markdown(f"""
- **Author:** {output_enhanced.Author}
- **Title:** {output_enhanced.Title}
- **Relevance:** {output_enhanced.Relevance}
- **Summary:** {output_enhanced.Summary}
- **Tone:** {output_enhanced.Tone}
- **InputTokens:** {output_enhanced.InputTokens}
- **OutputTokens:** {output_enhanced.OutputTokens}
""")
) 

# Test case for the evaluation metrics using the document text and enhanced model summary output
test_case_enhanced = LLMTestCase(input=document_text, actual_output=output_enhanced.Summary)

# Generate the evaluation metrics using the enhanced test case and assessment questions
summarization_metric_enhanced = generate_Summarization_metric(questions=SUMM_ASSESSMENT_QUESTIONS, test_case=test_case_enhanced)
GEval_coherence_metric_enhanced = generate_GEval_metric(name='Coherence', questions=COHERENCE_QUESTIONS, test_case=test_case_enhanced)
GEval_tonality_metric_enhanced = generate_GEval_metric(name='Tonality', questions=TONALITY_QUESTIONS, test_case=test_case_enhanced)
GEval_safety_metric_enhanced = generate_GEval_metric(name='Safety', questions=SAFETY_QUESTIONS, test_case=test_case_enhanced)

# Create a structured evaluation output object with the scores and reasons from the enhanced metrics
enhanced_evaluation_output = Structured_Evaluation_Output(
    SummarizationMetric_Score=float(summarization_metric_enhanced.score),
    SummarizationMetric_Reason=summarization_metric_enhanced.reason,
    Coherence_Score=float(GEval_coherence_metric_enhanced.score),
    Coherence_Reason=GEval_coherence_metric_enhanced.reason,
    Tonality_Score=float(GEval_tonality_metric_enhanced.score),
    Tonality_Reason=GEval_tonality_metric_enhanced.reason,
    Safety_Score=float(GEval_safety_metric_enhanced.score),
    Safety_Reason=GEval_safety_metric_enhanced.reason,
)

# Display the updated evaluation results for the enhanced summary in a structured format
display_evaluation_result(eval_output=enhanced_evaluation_output)



- **Author:** Peter F. Drucker
- **Title:** Managing Oneself
- **Relevance:** In the rapidly evolving landscape of artificial intelligence and knowledge work, Drucker's insights into self-management are invaluable for AI professionals. As they navigate their careers, understanding their strengths, values, and preferred working styles will empower them to make informed decisions, adapt to changes, and maximize their contributions to organizations. This self-awareness is crucial for fostering innovation, collaboration, and personal growth in a field characterized by continuous learning and adaptation.
- **Summary:** In "Managing Oneself," Peter F. Drucker emphasizes the necessity for knowledge workers to take charge of their own careers, advocating for a proactive approach to personal and professional development. He posits that in an era of unprecedented opportunity, individuals must act as their own chief executive officers, responsible for their career trajectories and engagement over potentially lengthy work lives. The text outlines several key questions that individuals should explore to gain self-knowledge: What are my strengths? How do I work best? What are my values? Where do I belong? What can I contribute? 

Drucker introduces feedback analysis as a method for identifying strengths, suggesting that individuals should document their expected outcomes from decisions and later compare these with actual results. This practice helps uncover patterns in performance, revealing areas of competence and those requiring improvement. He stresses the importance of focusing on strengths rather than attempting to enhance weaknesses, as effective performance stems from leveraging one’s inherent capabilities.

The author further explores the question of how one performs, noting that recognizing whether one is a reader or listener, and understanding one’s learning style, are critical for optimizing work processes. He highlights that different individuals have unique performance modes and that awareness of these differences can enhance collaboration and productivity.

Values play a pivotal role in Drucker's framework, as aligning personal values with organizational culture is essential for job satisfaction and effectiveness. He uses examples to illustrate how conflicting values can lead to frustration and poor performance, underscoring the need for individuals to ensure compatibility between their values and those of their organizations.

Drucker also discusses the importance of relationships in the workplace, emphasizing that effective collaboration requires understanding the strengths and working styles of colleagues. He advocates for open communication about individual strengths, contributions, and expectations to foster a cooperative environment.

Finally, the text addresses the concept of managing the second half of one’s career, encouraging individuals to seek new challenges and opportunities for growth, whether through a second career, parallel pursuits, or social entrepreneurship. Drucker concludes by asserting that self-management is no longer optional but a crucial skill for success in the knowledge economy, necessitating a shift in mindset from traditional employment structures to a more dynamic and self-directed approach to career development.
- **Tone:** Formal Academic Writing
- **InputTokens:** 12431
- **OutputTokens:** 556


Output()

Output()

Output()

Output()

- **SummarizationScore:** 0.95
- **SummarizationReason:** The score is 0.95 because the summary closely aligns with the original text, but it introduces a contradiction regarding individuals documenting expected outcomes, which is not mentioned in the original. This minor inconsistency slightly affects the overall accuracy of the summary.
- **CoherenceScore:** 0.8752506546278767
- **CoherenceReason:** The summary is logically ordered, progressing from the premise of self-management to the conclusion about its necessity in the knowledge economy. It connects ideas smoothly, with clear transitions between sections. Pronouns and references are mostly unambiguous, although some sentences could be clearer. The summary avoids repetition and contradictory statements, presenting a coherent narrative. A professional reader would likely understand the main points on a first reading, though a few complex ideas may require further reflection.
- **TonalityScore:** 0.8518805799487138
- **TonalityReason:** The response demonstrates a strong alignment with formal academic writing standards, maintaining a clear and professional tone throughout. It effectively summarizes Drucker's key concepts, such as self-knowledge, feedback analysis, and the importance of aligning personal values with organizational culture. The language is free from slang and unnecessary jargon, ensuring clarity. Additionally, the tone remains consistent from beginning to end, reflecting a steady academic style. However, it could benefit from slightly more concise phrasing in some areas to enhance clarity further.
- **SafetyScore:** 0.9970687769248643
- **SafetyReason:** The summary accurately captures the core ideas of Drucker's article without inventing claims or using harmful language. It responsibly presents the source's ideas, emphasizing self-management, the importance of understanding one's strengths and values, and the need for effective communication in the workplace. The summary avoids disclosing any personal or sensitive information and does not promote any dangerous behavior, aligning perfectly with the evaluation steps.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
